# Imaging: reward-aligned responses and most responsive cells

Everything is on **pyControl time (s)**, using the same rsync alignment as the photometry, while keeping **frame numbers**: frame `i` of `F.npy` has pyControl time `time_s[i]` in `run-XX/frame_times.npz`.

1. **Frame times.** For each run, match the raw TIFF, the acquisition's DAQ file (`file_*.h5`: frame clock and rsync pulses) and the behaviour `.tsv` whose rsync pulses align with the DAQ's, and save `frame_times.npz` beside the run in derivatives. Written once per run; both destriping variants share it.
2. **Reward-aligned responses.** Per variant: each cell's mean response around the outcome (reward onset, or no-reward cue onset for 0 pulses) per number of reward pulses, as robust z-scores (default) or raw F and ΔF/F0 (`MEASURES`) of the green and red channels; then mean ± SEM across all cells of all runs.
3. **Most responsive cells.** Per variant and run: the `N_TOP` best cells by Suite2p's classifier probability p(cell) and, separately, by largest green ΔF/F0, their traces with the outcomes marked, and where they are.

Needs the extracted planes (batch + Cellpose extraction in the mini2p repo) and `uv sync --extra funcimg --extra notebooks`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages
from mini2p_preprocessing.progress import ProgressLog

import src.behavior_visualization.plot_style  # noqa: F401
from src.behavior_analysis.outcome_events import OutcomeParseError
from src.behavior_import.session_order import session_number
from src.funcimg_analysis.collect import collect_pulse_responses, top_cells, variant_runs
from src.funcimg_analysis.frame_times import (
    FRAME_TIMES_FILE,
    RunMismatchError,
    load_frame_times,
    write_frame_times,
)
from src.funcimg_import.runs import session_run_files
from src.funcimg_visualization.plot_funcimg_responses import (
    plot_pulse_responses,
    plot_top_cells,
    top_cells_pdf_name,
)
from src.settings import REPO_ROOT, get_settings

EXPERIMENT = "3x3_field_magnitude_bandit"
COHORT = "cohort-01"
SUBJECT = "sub-03_id-MY_71_R"
SESSION_FIRST, SESSION_LAST = 224, 224  # one session: give the same number twice
RUNS = ("run-01",)  # every run; or only these runs of each session, e.g. ("run-02",)
SOURCE_SUBSTRING = "71rtask"  # picks the imaging TIFFs, as in the batch preprocessing
VARIANTS = ("without_destriping", "with_destriping")
WINDOW_S = (-10.0, 30.0)  # around the outcome; rounded to whole frames
PHASES = ("task",)  # add "port_finding" to include port-finding trials
X_UNITS = "s"  # "s" or "frames", for every figure
MEASURES = ("F z", "dF/F0 z")  # robust z, baseline-subtracted; or ("F", "dF/F0") for raw
# z needs frames before the outcome: with a WINDOW_S starting at 0, use the raw measures
MEASURE_TAG = "-".join(m.replace("/", "").replace(" ", "") for m in MEASURES)  # in file names
CLIP_PERCENTILES = (1, 99)  # top-cell traces clipped to this range per ROI; None = unclipped
N_TOP = 30
RANKINGS = ("p_cell", "max_dff")  # one PDF per ranking
OVERWRITE_FRAME_TIMES = False

data_root = get_settings().data_root
RAW = data_root / EXPERIMENT / "rawdata" / COHORT / SUBJECT
DERIVATIVES = data_root / EXPERIMENT / "derivatives" / COHORT / SUBJECT
FIGURES = REPO_ROOT / "results" / "figures" / "funcimg" / COHORT / SUBJECT
FIGURES.mkdir(parents=True, exist_ok=True)
SCOPE = f"sessions {SESSION_FIRST}-{SESSION_LAST}" + (f", {', '.join(RUNS)}" if RUNS else "")
FILE_TAG = f"ses-{SESSION_FIRST}-{SESSION_LAST}" + (f"_{'_'.join(RUNS)}" if RUNS else "")
# Folders without a session number (e.g. ses-baseplating) are skipped.
sessions = sorted(
    (
        p
        for p in RAW.glob("ses-*")
        if session_number(p.name) is not None
        and SESSION_FIRST <= session_number(p.name) <= SESSION_LAST
        and (p / "funcimg").is_dir()
    ),
    key=lambda p: session_number(p.name),
)
print(f"{len(sessions)} imaging sessions in {RAW}")

## 1. Frame times

Writes `derivatives/.../funcimg/run-XX/frame_times.npz` for every run the batch has preprocessed. Before writing, the run's TIFF is checked against the batch's own `source_tiff.json`, so a run folder can never be paired with another acquisition's frame clock or behaviour. A frame's time is the start of its acquisition (frame-clock rising edge). Existing files are kept unless `OVERWRITE_FRAME_TIMES`. The table shows how well each run's rsync pulses matched; `fraction_matched` should be close to 1 and the interval discrepancy a few ms at most.

In [ ]:
rows, problems = [], []
progress = ProgressLog("Frame times", len(sessions), print)
for session_dir in sessions:
    try:
        runs = session_run_files(session_dir, SOURCE_SUBSTRING)
        if RUNS is not None:
            runs = [run for run in runs if run.run_name in RUNS]
    except ValueError as error:
        problems.append((session_dir.name, "-", str(error)))
        progress.step_done(succeeded=False)
        continue
    for run in runs:
        run_dir = DERIVATIVES / session_dir.name / "funcimg" / run.run_name
        path = run_dir / FRAME_TIMES_FILE
        try:
            if OVERWRITE_FRAME_TIMES or not path.exists():
                write_frame_times(run, run_dir, overwrite=OVERWRITE_FRAME_TIMES)
            saved = load_frame_times(run_dir)
        except (RunMismatchError, FileNotFoundError) as error:
            problems.append((session_dir.name, run.run_name, str(error)))
            continue
        rows.append({
            "session": session_dir.name, "run": run.run_name, "tiff": run.tiff.name,
            "daq": run.h5.name, "behaviour": run.tsv.name, "frames": len(saved.time_s),
            "frames_with_time": int(pd.notna(saved.time_s).sum()),
            "fraction_matched": round(saved.quality["fraction_matched"], 3),
            "max_interval_discrepancy_ms": saved.quality["max_abs_interval_discrepancy_ms"],
        })
    progress.step_done()
display(pd.DataFrame(rows))
for problem in problems:
    print("NOT WRITTEN:", *problem)

## 2. Reward-aligned responses, by number of reward pulses

0 pulses is aligned to the no-reward cue onset, 1-4 pulses to reward onset (different events). The outcome goes to the frame whose start is nearest it (within half a frame), so time 0 is that frame's start. "Cells" are all Cellpose ROIs; Suite2p's `iscell` classifier is not used to exclude any. Each cell is averaged over its trials first, so each cell counts once; the shading is the SEM across cells. The default views are robust z-scores (each cell's whole trace: (x - median) / (1.4826 · MAD)) with each trial's mean over the part of the window before the outcome subtracted, so cells with the same trend but different raw F or ΔF/F0 scale do not average out; a cell with MAD 0 has no scale, is left out and is listed. Set `MEASURES = ("F", "dF/F0")` for the raw views (raw F reflects the cells' brightness). Runs left out are listed with the reason.

In [ ]:
responses = {}
for variant in VARIANTS:
    pooled, runs, skipped = collect_pulse_responses(
        DERIVATIVES, SESSION_FIRST, SESSION_LAST, variant, WINDOW_S, PHASES, runs=RUNS,
        measures=MEASURES,
    )
    responses[variant] = (pooled, runs)
    print(f"{variant}: {len(runs)} runs, {sum(r['n_cells'] for r in runs)} cells")
    for r in runs:
        for view, cells in r["flat_cells"].items():
            if cells:
                where = f"{r['session']} {r['run']} {view}"
                print(f"  {where}: ROIs {cells} have MAD 0; z is NaN")
    for session, run, reason in skipped:
        print(f"  LEFT OUT {session} {run}: {reason}")
    if not runs:
        print(f"  {variant}: nothing to plot")
        continue
    counts = pd.DataFrame([{"session": r["session"], "run": r["run"], "cells": r["n_cells"],
                            **{f"{p} pulses": r["n_trials"].get(p, 0) for p in range(5)},
                            "outcomes dropped": r["n_events_dropped"]} for r in runs])
    display(counts)
    fig = plot_pulse_responses(
        pooled, runs[0]["offsets"], runs[0]["period_s"], x_units=X_UNITS,
        measures=MEASURES,
        title=f"{SUBJECT}, {SCOPE}, {variant}: mean ± SEM across cells",
    )
    name = f"pulse_responses_{FILE_TAG}_{variant}_{MEASURE_TAG}_{X_UNITS}.png"
    fig.savefig(FIGURES / name, dpi=200)
    plt.show()

## 3. The most responsive cells of each run

Two rankings, each its own PDF with one page per run: `top{N_TOP}_cells_by_p_cell_….pdf` ranks cells by Suite2p's classifier probability p(cell) (`iscell.npy` column 1; advisory, since Suite2p's built-in classifier is run on Cellpose ROIs, not tuned for them), and `top{N_TOP}_cells_by_max_dff_….pdf` by the largest green ΔF/F0 anywhere in the recording (a single bright frame can rank a cell high, so check the traces). Ties go to the lower ROI index; each trace's label shows its p(cell). Each trace is drawn clipped to its own `CLIP_PERCENTILES` range (display only; the ranking uses the unclipped values), and the rows are spaced so the clipped traces never overlap. The first run of each is shown here.

In [ ]:
for variant in VARIANTS:
    found = variant_runs(DERIVATIVES, SESSION_FIRST, SESSION_LAST, variant, RUNS)
    if not found:
        print(f"{variant}: no registered runs found")
        continue
    for rank_by in RANKINGS:
        progress = ProgressLog(f"Top cells by {rank_by} ({variant})", len(found), print, 30)
        pdf_path = FIGURES / top_cells_pdf_name(N_TOP, rank_by, FILE_TAG, variant, X_UNITS)
        pdf = None  # opened on the first page, so a ranking with no plottable run writes no PDF
        for session, run_dir, plane_dir in found:
            try:
                top = top_cells(plane_dir, run_dir, N_TOP, phases=PHASES, rank_by=rank_by)
            except (RunMismatchError, OutcomeParseError, FileNotFoundError) as error:
                print(f"  LEFT OUT {session} {run_dir.name}: {error}")
                progress.step_done(succeeded=False)
                continue
            fig = plot_top_cells(
                top, x_units=X_UNITS, title=f"{session} {run_dir.name}", variant=variant,
                clip_percentiles=CLIP_PERCENTILES,
            )
            if pdf is None:
                pdf = PdfPages(pdf_path)
                display(fig)  # the first run, here; every run is in the PDF
            pdf.savefig(fig)
            plt.close(fig)
            progress.step_done()
        if pdf is None:
            print(f"{variant}: no run could be plotted by {rank_by}")
            continue
        pdf.close()
        print("Saved", pdf_path)
